#### CREATE `CUSTOMERS_VW` FROM VOLUME
- CATALOG NAME: GIZMO
- SCHEMA NAME: BRONZE
- VIEW NAME: CUSTOMERS_VW

In [0]:
%python
customers_df = (spark.read.format('json').load('/Volumes/gizmo/landing/operational_data/customers/'))
# display(customers_df.limit(10))
print(f'Row Count: {customers_df.count()}')

In [0]:
CREATE OR REPLACE VIEW GIZMO.BRONZE.CUSTOMERS_VW
AS
SELECT
customer_id,
customer_name,
to_date(date_of_birth,'yyyy-MM-dd') as date_of_birth,
email,
to_date(member_since,'yyyy-MM-dd') as member_since,
telephone,
to_timestamp(created_timestamp,'yyyy-MM-dd HH:mm:ss') as created_timestamp,
_metadata.file_path as file_path,
_metadata.file_name as file_name,
current_timestamp() as load_timestamp
 FROM json.`/Volumes/gizmo/landing/operational_data/customers/`;

#### QUERY `CUSTOMERS_VW` TO VALIDATE THE DATA

In [0]:
SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW;

In [0]:
%python
customers_count_df = spark.sql('''SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW''');
print(f'Row Count: {customers_count_df.count()}')

#### BELOW COMMAND TO EXECUTE THE FUNCTION

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config

In [0]:
%python
try:
    verify_pipeline_counts(customers_df, customers_count_df, "01.IngestCustomersJSON")
except AssertionError as e:
    # This ensures the notebook actually fails if scheduled via a Databricks Workflow/Job
    raise e

In [0]:
%skip
%python
spark.sql("""
CREATE OR REPLACE TABLE GIZMO.BRONZE.INGEST_LOGS (
  log_id STRING,
  event_time TIMESTAMP,
  event_type STRING,
  source_table STRING,
  target_table STRING,
  record_count BIGINT,
  status STRING,
  message STRING,
  user_name STRING,
  notebook_path STRING,
  pipeline_name STRING
)
COMMENT 'Logging table for data ingestion events and pipeline activity'
""")

## 📝 CAPTURE AUDIT & OBSERVABILITY MECHANISM

Track and log every data pipeline run for transparency, traceability, and operational monitoring.  
This section ensures all data loads are auditable and pipeline health is observable.

In [0]:
%skip
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import pyspark.sql.functions as F

# 1. Timestamps & Dates
if 'load_start_time' not in locals():
    load_start_time = datetime.now() 

load_end_time = datetime.now()
current_date = load_end_time.date() # YYYY-MM-DD

# ---------------------------------------------------------
# NEW LOGIC: Calculate Sequential Run ID for Today
# ---------------------------------------------------------
try:
    # Query the audit table for the max run_id logged today
    max_run_df = spark.table("GIZMO.BRONZE.AUDIT_LOGS") \
        .filter(F.col("event_time") == current_date) \
        .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
    
    max_id_row = max_run_df.collect()[0]
    
    if max_id_row["max_id"] is not None:
        next_run_int = max_id_row["max_id"] + 1
    else:
        next_run_int = 1
        
except Exception as e:
    next_run_int = 1

# Format the integer as a 2-digit string with leading zeros
run_id_str = f"{next_run_int:02d}"
# ---------------------------------------------------------

# 2. Grab Databricks environment metadata safely (FIXED)
try:
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_path = context.notebookPath().get() # FIXED: Avoided redundant call chain breakdown
    user_name = context.tags().apply("user")
    pipeline_name = '01.IngestCustomersJSON' 
except Exception:
    notebook_path = "Unknown/Local"               # FIXED: Changed from hardcoded string of code
    user_name = "System"
    pipeline_name = '01.IngestCustomersJSON'       # FIXED: Changed from self-referential assignment

# 3. Extract your data count
record_count = customers_count_df.count()

# 4. Construct the row matching your specific DDL column types
log_entry = Row(
    log_id=str(uuid.uuid4()),
    run_id=run_id_str,                
    event_time=current_date,          
    event_type="FULL LOAD",
    source_table="json.`/Volumes/gizmo/landing/operational_data/customers/`",
    target_table="GIZMO.BRONZE.CUSTOMERS_VW",
    record_count=record_count,
    status="SUCCESS",
    message="Loaded customers data into bronze view",
    user_name=user_name,
    notebook_path=notebook_path,
    pipeline_name=pipeline_name,
    load_start_time=load_start_time,  
    load_end_time=load_end_time       
)

# 5. Define the strict schema mapping
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

log_schema = StructType([
    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),  
    StructField("event_time", DateType(), True),
    StructField("event_type", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("load_start_time", TimestampType(), True),
    StructField("load_end_time", TimestampType(), True)
])

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE PIPELINE METADATA
# =====================================================================
# Capture start time at the absolute beginning of the execution
load_start_time = datetime.now()
pipeline_name = '01.IngestCustomersJSON'

# Establish default tracking states
status = "SUCCESS"
message = "Loaded customers data into bronze view"
record_count = 0

try:
    # =====================================================================
    # 2. CORE ETL LOGIC
    # =====================================================================
    
    # Step A: Extract Data using absolute path and explicit format configuration
    customers_df = (spark.read
                    .format("json")
                    .load("/Volumes/gizmo/landing/operational_data/customers/"))
    
    # Step B: Execute target transformations or loading actions here
    # (Example: customers_df.write.mode("overwrite").saveAsTable("GIZMO.BRONZE.CUSTOMERS"))
    
    # Step C: Capture final evaluated source record count
    record_count = customers_df.count()
    
    # =====================================================================

except Exception as e:
    # 3. EXCEPTION HANDLING
    # If any error occurs above, catch it, flip status, and parse the trace
    status = "FAILED"
    
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1  # Standard indicator flag representing an uncompleted execution

finally:
    # =====================================================================
    # 4. AUDIT & LOGGING (Guaranteed execution via finally block)
    # =====================================================================
    load_end_time = datetime.now()
    current_date = load_end_time.date()

    # Step A: Calculate Sequential Run ID for Today
    try:
        max_run_df = spark.table("GIZMO.BRONZE.AUDIT_LOGS") \
            .filter(F.col("event_time") == current_date) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception:
        # Defaults to 1 if table is empty or uninitialized
        next_run_int = 1

    # Apply 2-digit zero padding format string (e.g., 1 -> "01", 11 -> "11")
    run_id_str = f"{next_run_int:02d}"

    # Step B: Secure Notebook Cluster Context Metadata safely
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"

    # Step C: Package the metadata tracking Row
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table="json.`/Volumes/gizmo/landing/operational_data/customers/`",
        target_table="GIZMO.BRONZE.CUSTOMERS_VW",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )

    # Step D: Declare structured explicit Schema types matching Target DDL exactly
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])

    # Step E: Instantiate log DataFrame and append transactional trace record
    log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
    log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.BRONZE.AUDIT_LOGS")
    print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")

    # Step F: Force a hard stop exception for workflow orchestrators if pipeline failed
    if status == "FAILED":
        raise RuntimeError(message)

#### CONVERT TO DATAFRAME AND APPEND TO DELTA TABLE

In [0]:
%skip
%python
# 6. Convert to DataFrame and append to Delta
log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.BRONZE.AUDIT_LOGS")

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS HAS BEEN LOADED SUCCESSFULLY INTO GIZMO.BRONZE.CUSTOMERS_VW")

In [0]:
SELECT * FROM GIZMO.BRONZE.AUDIT_LOGS ORDER BY run_id DESC;

In [0]:
%sql
DROP TABLE GIZMO.BRONZE.AUDIT_LOGS